# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eman123-123/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
*Classification, clustering, ranking, or scoring — which one, and why?*

**Classification.** I want to classify pages as declining or not declining using observable page, search, and content signals. The prediction supports a review decision: which pages should be investigated first when review capacity is limited. Classification fits because the starter task provides a yes/no proxy label for declining pages.

In [18]:
import pandas as pd

raw_path = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 30,000
Columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target/proxy:** whether a page is declining.

For the starter dataset, I define the proxy label as `trend_direction == "down"`. This label is derived from the observed 30-day versus previous 30-day impression trend, so it is a current-window proxy rather than a future outcome. It is useful for this assignment, but it should not be treated as proof that a page will continue declining or that the decline was caused by any particular factor.

In [19]:
df["target_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

df[["trend_direction", "target_declining"]].head(10)

,trend_direction,target_declining
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50.** I will measure how many of the top 50 pages selected for review are actually labeled as declining. A higher Precision@50 means the limited review queue contains more pages matching the decline proxy. This metric fits the decision because the practical goal is to prioritize a small number of pages for human investigation.

In [20]:
label_rate = df["target_declining"].mean()

print(f"Declining proxy rate: {label_rate:.3f}")
print(f"Expected positives in 50 at base rate: {label_rate * 50:.1f}")

Declining proxy rate: 0.542
Expected positives in 50 at base rate: 27.1


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one page-level content item.**

Each row contains observable page, search, and content signals, including impressions, clicks, CTR, average position, content age, and update age. The page identifier is pseudonymous. I will use these signals as inputs and keep the decline label separate from the features.

In [21]:
page_cols = [
    "content_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "target_declining",
]

page_df = df[page_cols].copy()

page_df.head(10)

,content_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,trend_direction,target_declining
0,content_304f48230142,keyword article,3803,29,0.76,10.6,187,20,down,1
1,content_a1fb4e703a9e,keyword article,15320,7,0.05,20.3,445,25,down,1
2,content_9aa793d4d895,keyword article,12581,11,0.09,36.5,141,20,down,1
3,content_331d6c4de07b,keyword article,11751,58,0.49,6.2,463,22,stable,0
4,content_d99b7a2d90ca,keyword article,19140,24,0.13,44.0,263,14,down,1
5,content_d4084a4bc775,keyword article,3970,1,0.03,8.5,147,20,down,1
6,content_9a34b442b552,keyword article,20,0,0.00,7.0,90,20,down,1
7,content_a63219c6e95a,keyword article,1724,1,0.06,21.2,445,22,stable,0
8,content_5e6c160719bc,keyword article,32574,29,0.09,46.0,90,20,down,1
9,content_c27558df2b0c,keyword article,1240,2,0.16,4.9,257,104,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could flag pages using one threshold, such as a minimum impression count or a single trend signal. That would be easy to explain, but page-level decline patterns can involve several observable signals at the same time, including search volume, impressions, CTR, average position, content age, and update age.

A classification model can combine several signals and learn interactions between them, then produce a probability that can be used to prioritize a review queue. The benefit is decision support, not proof of causation: the model may help identify pages worth investigating first, but it cannot prove why a page declined or guarantee that a refresh will recover it.

In [22]:
signal_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]

df[signal_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.